In [ ]:
# ── 0A  Install dependencies ────────────────────────────────────────────
import subprocess
import sys
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# Torch — uncomment the line that matches your hardware:
pip('torch','torchvision','--index-url','https://download.pytorch.org/whl/rocm6.4')  # AMD ROCm 6.4
# pip('torch','torchvision','--index-url','https://download.pytorch.org/whl/rocm6.2')  # AMD ROCm 6.2
# pip('torch','torchvision','--index-url','https://download.pytorch.org/whl/cu124')    # NVIDIA CUDA 12.4

# diffusers from source (ZImagePipeline not yet on PyPI)
pip('git+https://github.com/huggingface/diffusers')

# core runtime
pip('safetensors>=0.7.0','accelerate>=1.0.0','transformers>=5.0.0',
    'peft>=0.18.0','huggingface_hub>=1.9.0',
    'datasets>=3.0.0','pandas>=2.0.0',
    'matplotlib>=3.9.0','ipywidgets>=8.0.0',
    'optimum-quanto>=0.2.0','Pillow>=10.0.0')

print('All packages installed.')

# support CJK characters in matplotlib
import os
import urllib.request

font_url = "https://github.com/googlefonts/noto-cjk/raw/main/Sans/OTF/SimplifiedChinese/NotoSansCJKsc-Regular.otf"
font_path = "NotoSansCJKsc-Regular.otf"

if not os.path.exists(font_path):
    urllib.request.urlretrieve(font_url, font_path)

from matplotlib import font_manager, rcParams
font_manager.fontManager.addfont(font_path)

prop = font_manager.FontProperties(fname=font_path)
font_name = prop.get_name()

rcParams["font.family"] = font_name
rcParams["axes.unicode_minus"] = False

In [ ]:
# ── 0B  HF Auth + Download model + dataset in one shot ──────────────────
import os
from huggingface_hub import login, snapshot_download, hf_hub_download
import torch

# MANUALLY ADD YOUR TOKEN HERE
HF_TOKEN = '___'  # <- Replace with your real token
login(token=HF_TOKEN)
os.environ['HF_TOKEN'] = HF_TOKEN

MODEL_REPO   = 'DownFlow/Z-Image-Turbo-Fuli'
DATASET_REPO = 'DownFlow/fuliji'
PARQUET_FILE = 'dataset.parquet'

print(f'[1/1] Downloading {MODEL_REPO}  (~20 GB, cached on repeat runs) ...')
MODEL_DIR = snapshot_download(MODEL_REPO, token=HF_TOKEN)
print(f'  -> {MODEL_DIR}')

# GPU check
if torch.cuda.is_available():
    d = torch.cuda.get_device_properties(0)
    print(f'GPU: {d.name}  {d.total_memory/2**30:.1f} GB VRAM')
else:
    print('WARNING: no CUDA GPU detected.')
print('Setup complete.')